# Notebook 1: 01_Data_Foundation

This notebook establishes the structural integrity of the pipeline. It handles data ingestion, cleans UTC timestamps, runs an automated QA check, engineers physical traffic features, and exports the clean data into a `.parquet` format.

### Cell 1: Project Setup & Directory Configuration

In [ ]:
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Initializing Gridlock Data Foundation...")

# Build the production directory structure
directories = [
    "data/raw",
    "data/processed",
    "models",
    "notebooks"
]

for dir_path in directories:
    Path(dir_path).mkdir(parents=True, exist_ok=True)

print("✅ Directory structure validated.")


Initializing Gridlock Data Foundation...
✅ Directory structure validated.


### Cell 2: Data Loading & Temporal Engineering

In [ ]:
# Downloads the dataset automatically behind the scenes
!gdown "https://drive.google.com/uc?id=1h9IVhvX45W-TPGmqV9Mwf8ue9Rb1w2Cz" -O gridlock_data.csv

DATASET_PATH = "/content/gridlock_data.csv"

Downloading...
From (original): https://drive.google.com/uc?id=1h9IVhvX45W-TPGmqV9Mwf8ue9Rb1w2Cz
From (redirected): https://drive.google.com/uc?id=1h9IVhvX45W-TPGmqV9Mwf8ue9Rb1w2Cz&confirm=t&uuid=eb976ff0-e830-49fa-a112-8b2916ec2125
To: /content/gridlock_data.csv
100% 110M/110M [00:01<00:00, 91.5MB/s]


In [ ]:
# # OR , Manually Upload dataset
# from google.colab import files
# import pandas as pd

# # Upload dataset
# uploaded = files.upload()

# # Get uploaded file name
# DATASET_PATH = next(iter(uploaded))

# # Read CSV
# df = pd.read_csv(DATASET_PATH)

# # Preview
# print(f"Loaded: {DATASET_PATH}")
# df.head()

In [ ]:
print("Loading raw dataset...")
df = pd.read_csv(DATASET_PATH)
initial_rows = len(df)

# 1. Datetime Parsing (Handling mixed microseconds and UTC offsets)
df["created_datetime"] = pd.to_datetime(df["created_datetime"], format='mixed', utc=True)

# 2. Convert to Indian Standard Time (IST)
df["created_ist"] = df["created_datetime"].dt.tz_convert("Asia/Kolkata")

# 3. Extract Time Features
df["hour"] = df["created_ist"].dt.hour
df["dow"] = df["created_ist"].dt.dayofweek
df["date"] = df["created_ist"].dt.date
df["month"] = df["created_ist"].dt.month

print(f"✅ Loaded {initial_rows} rows. Temporal features engineered.")

Loading raw dataset...
✅ Loaded 298450 rows. Temporal features engineered.


### Cell 3: Automated QA & Coordinate Validation

In [ ]:
# CELL 3: DATA QUALITY ASSURANCE (UPDATED)
print("Running Data QA Protocol...")

qa_report = {}
qa_report["Total Initial Rows"] = initial_rows
qa_report["Missing Coordinates"] = df[["latitude", "longitude"]].isna().sum().sum()

# Drop completely missing coordinates, but keep ALL valid spatial data
df = df.dropna(subset=["latitude", "longitude"])

# Deduplication (If the same device fired multiple times in the same second)
duplicates = df.duplicated(subset=["device_id", "created_ist"]).sum()
qa_report["Duplicate Event Triggers"] = duplicates
df = df.drop_duplicates(subset=["device_id", "created_ist"])

print("--- QA REPORT ---")
for key, val in qa_report.items():
    print(f"{key}: {val}")
print(f"✅ Retained {len(df)} clean records.")

Running Data QA Protocol...
--- QA REPORT ---
Total Initial Rows: 298450
Missing Coordinates: 0
Duplicate Event Triggers: 114039
✅ Retained 184411 clean records.


### Cell 4: Feature Engineering (Physical Congestion Impact)

In [ ]:
print("Engineering Congestion Impact Features...")

# 1. Vehicle Footprint Extraction
vehicle_weights = {
    "SCOOTER": 10, "PASSENGER AUTO": 15, "CAR": 20,
    "MAXI-CAB": 25, "TANKER": 35, "TRUCK": 40
}
# Fallback to updated_vehicle_type if available, else use vehicle_type
df["true_vehicle_type"] = df["updated_vehicle_type"].fillna(df["vehicle_type"])
df["vehicle_footprint"] = df["true_vehicle_type"].map(vehicle_weights).fillna(15)

# # 2. Road Capacity NLP Estimator
# def estimate_road_capacity(location_text, violation_str):
#     location_text, violation_str = str(location_text).upper(), str(violation_str).upper()
#     if "MAIN ROAD" in violation_str: return 100
#     if re.search(r'\b(MAIN ROAD|RING ROAD|HIGHWAY|ORR)\b', location_text): return 100
#     if re.search(r'\b(CROSS|LAYOUT|LANE|STREET|GULLY)\b', location_text): return 40
#     return 60

# df["road_capacity"] = df.apply(lambda x: estimate_road_capacity(x["location"], x["violation_type"]), axis=1)

# 2. Road Capacity NLP Estimator (Vectorized)
loc_str = df["location"].astype(str).str.upper()
viol_str = df["violation_type"].astype(str).str.upper()

# Define boolean masks for conditions
cond_main_viol = viol_str.str.contains("MAIN ROAD", na=False)
cond_main_loc = loc_str.str.contains(r'\b(MAIN ROAD|RING ROAD|HIGHWAY|ORR)\b', regex=True, na=False)
cond_cross = loc_str.str.contains(r'\b(CROSS|LAYOUT|LANE|STREET|GULLY)\b', regex=True, na=False)

# Apply logic: if Main/Highway = 100, if Cross/Lane = 40, else = 60
df["road_capacity"] = np.select(
    [cond_main_viol | cond_main_loc, cond_cross],
    [100, 40],
    default=60
)

# 3. Junction Impact Multiplier
df["is_junction"] = np.where(df["junction_name"] != "No Junction", 2.0, 1.0)

# 4. Final Base Impact Score (Footprint * Junction Penalty)
df["base_impact_score"] = df["vehicle_footprint"] * df["is_junction"]

print("✅ Physical impact modeling complete.")


Engineering Congestion Impact Features...
✅ Physical impact modeling complete.


### Cell 5: Reporting Bias & System Confidence

In [ ]:
print("Analyzing System Reporting Bias...")

# Calculate volume by hour of day
hourly_stats = df.groupby("hour").size().reset_index(name="total_violations")

# Define peak baseline as the hour with maximum data flow
peak_baseline = hourly_stats["total_violations"].max()

# Calculate Confidence %
hourly_stats["reporting_confidence"] = (hourly_stats["total_violations"] / peak_baseline).round(3)

# Merge back to main dataframe
df = df.merge(hourly_stats[["hour", "reporting_confidence"]], on="hour", how="left")

print("✅ Reporting Bias quantified. Hourly stats generated.")
display(hourly_stats.sort_values('hour').head(5))

Analyzing System Reporting Bias...
✅ Reporting Bias quantified. Hourly stats generated.


,hour,total_violations,reporting_confidence
0,0,3159,0.154
1,1,6731,0.328
2,2,10523,0.513
3,3,13468,0.656
4,4,14065,0.685


### Cell 6: Data Export & Pipeline Checkpointing

In [ ]:
print("Exporting Foundation Artifacts...")

# 1. Main Cleaned Dataset (Parquet for high performance)
CLEANED_DATA_PATH = "data/processed/cleaned_dataset.parquet"
df.to_parquet(CLEANED_DATA_PATH, index=False)

# 2. Analytics Tables (CSVs for easy review/dashboarding)
hourly_stats.to_csv("data/processed/reporting_bias.csv", index=False)

# Generate Daily Stats
daily_stats = df.groupby("date").size().reset_index(name="daily_volume")
daily_stats.to_csv("data/processed/daily_statistics.csv", index=False)

# Generate Vehicle Stats
vehicle_stats = df["true_vehicle_type"].value_counts().reset_index()
vehicle_stats.columns = ["Vehicle Type", "Total Violations"]
vehicle_stats.to_csv("data/processed/vehicle_statistics.csv", index=False)

# Save QA Report
pd.DataFrame(list(qa_report.items()), columns=["Metric", "Value"]).to_csv("data/processed/data_validation_report.csv", index=False)

print(f"🚀 SUCCESS: Foundation phase complete. Clean data saved to {CLEANED_DATA_PATH}")

Exporting Foundation Artifacts...
🚀 SUCCESS: Foundation phase complete. Clean data saved to data/processed/cleaned_dataset.parquet
